# Phase 6: Agent Loop Evaluation

This notebook verifies model availability and runs the Agent Loop over the dataset to measure:
- `task_completion_rate`: Tasks that successfully pass tests
- `recovery_rate`: Tasks that initially failed tests but recovered via agent tool actions (`write_file`, `run_tests`)
- `steps_to_success`: Average steps taken for successful completions

In [1]:
# 1. Verify Cached Models
import os

cache_dir = "D:/huggingface_cache/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a"
if os.path.exists(cache_dir):
    print("Qwen3.5-4B is verified and ready in local cache:", cache_dir)
    for fname in os.listdir(cache_dir):
        if fname.endswith((".safetensors", ".json")):
            print(f"  - {fname}")
else:
    print("Model snapshot not found in expected cache directory.")

Qwen3.5-4B is verified and ready in local cache: D:/huggingface_cache/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
  - config.json
  - model.safetensors-00001-of-00002.safetensors
  - model.safetensors-00002-of-00002.safetensors
  - model.safetensors.index.json
  - tokenizer.json
  - tokenizer_config.json
  - vocab.json


### 2. Run Agent Evaluation

We can evaluate either:
- `configs/agent_eval.yaml`: **2B SFT model** with fine-tuned coding ability.
- `configs/agent_eval_4b.yaml`: **4B Base model** to test zero-shot scaling.

All outputs will be saved to the root-level `experiments/` directory.

In [2]:
from llm_lab.agent.evaluator import evaluate_agent

# Choose configuration (path is automatically resolved to project root):
# config_path = "configs/agent_eval.yaml"       # Qwen3.5-2B SFT
config_path = "configs/agent_eval.yaml"

evaluate_agent(config_path)

d:\Github_Clones\local-llm-lab\.venv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
d:\Github_Clones\local-llm-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset from D:\Github_Clones\local-llm-lab\data\train_complex_500.jsonl...
Loading tokenizer Qwen/Qwen3.5-2B (cache_dir=D:/huggingface_cache)...
Loading model Qwen/Qwen3.5-2B (cache_dir=D:/huggingface_cache)...


Loading weights: 100%|██████████| 320/320 [00:01<00:00, 172.24it/s]


Loading LoRA adapter from D:\Github_Clones\local-llm-lab\experiments\exp01_sft_v1\checkpoint-final...
Running agent on 20 tasks...


100%|██████████| 20/20 [25:17<00:00, 75.86s/it] 


=== Agent Evaluation ===
Task Completion Rate: 20.0%
Avg Steps to Success: 5.0
Recovery Rate:        20.0%

Saved traces to D:\Github_Clones\local-llm-lab\experiments\exp05_agent_2b/agent_traces.jsonl


### 3. Display Metrics and Recovery Rate

In [ ]:
import json
import pprint
from llm_lab.constants import resolve_path

summary_path = resolve_path("experiments/exp05_agent_2b/summary.json")
try:
    with open(summary_path, 'r') as f:
        summary = json.load(f)
        print("Agent Evaluation Summary:")
        pprint.pprint(summary['metrics'])
except FileNotFoundError:
    print(f"Summary file not found at {summary_path}.")

Agent Evaluation Summary:
{'recovery_rate': 0.2, 'steps_to_success': 5.0, 'task_completion_rate': 0.2}


: 